<div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 40px; margin-top: 0;">
    <div style="flex: 0 0 auto; margin-left: 0; margin-bottom: 0; margin-top: 0;">
        <img src="./pics/UCSD Logo.png" alt="UCSD Logo" style="width: 200px; margin-bottom: 0px; margin-top: 20px;">
    </div>
    <div style="flex: 0 0 auto; margin-left: auto; margin-bottom: 0; margin-top: 20px;">
        <img src="./pics/wildfire.png" alt="WSTC Logo" style="width: 100px; margin-bottom: 0px;">
    </div>
    <div style="flex: 0 0 auto; margin-left: auto; margin-bottom: 0; margin-top: 20px;">
        <img src="./pics/sdsc-logo.png" alt="USGS Logo" style="width: 200px; margin-bottom: 0px;">
    </div>
</div>

<h1 style="text-align: center; font-size: 48px; margin-top: 0;">Shrub Mask Label Pipeline</h1>

In this notebook, we walk through a pipeline that turns TLS-detected shrubs into labeled NAIP masks. The end goal is simple: produce clean, georeferenced training labels that you can plug into your own shrub detection model.

## The Sites

We'll be working with a set of sites where TLS scans have already been collected and shrub lists are available. The sites are the following:

- [Calaveras Big Trees](https://www.parks.ca.gov/?page_id=551) 
- [DL Bliss](https://www.parks.ca.gov/?page_id=505) 
- [Independence Lake](https://www.fs.usda.gov/r05/tahoe/recreation/independence-lake)
- [Pacific Union College](https://www.puc.edu/about-puc/forest)
- [Sedgwick](https://sedgwick.nrs.ucsb.edu/)
- Shaver Lake

## The Data

Several data sources come together in this pipeline. All of them live in a shared cloud directory, organized by site, with each data product in its own subdirectory. The main root directory is `https://wifire-data.sdsc.edu/nc/public.php/dav/files` (you'll set this in Step 2).

#### ALS Files

Located in `ALS/`. We don't use the point cloud data itself here. Instead, we temporarily download each file just long enough to pull out the spatial metadata we need to anchor the TLS shrub locations in aerial coordinates.

#### Shrub Lists

Located in `shrub_lists/`. These are the heart of the pipeline — the detected shrubs we want to transfer onto NAIP imagery. If you need a refresher on how these lists are generated, head back to the *Shrub Lists Generation* module from Sprint 3.

**NOTE**: By default, this pipeline reads shrub lists produced by the original IntELiMon script. If yours were generated with the revised script, swap in the correct directory (look for the commented line at the top of the Step 9 function). When in doubt, revisit the *Shrub Lists Generation (Sprint 3)* module before continuing.

#### NAIP Imagery

Located in `NAIP_3DEP_product/`. These specific NAIP tiles were provided by the NASA WERK program. If you need a refresher on what NAIP is, head back to the *National Agriculture Imagery Program Demo* from Sprint 2.

#### Transformation Matrices

Located in `transformations/`. For each TLS file, there's a corresponding transformation matrix that encodes how to convert shrub coordinates from `TLS` space into `ALS` space.

## The Pipeline

Here's what happens under the hood, step by step:

1. Temporarily download `ALS` files and extract metadata with [**PDAL**]()
2. Temporarily download shrub lists and transformation matrices
3. Transform shrub coordinates into ALS space
4. Find the ALS tile that best matches each shrub list
5. Project shrub points into the NAIP CRS
6. Rasterize shrub points into a mask aligned to a tight crop over the NAIP grid
7. Save the mask

**NOTE**: As part of the final submission, teams are free to modify any part of this pipeline if they spot room for improvement. Make sure to document any changes in the final report.

## 1) Imports

In [ ]:
from __future__ import annotations

from collections import defaultdict
import json
import math
import re
import shutil
import subprocess
import tempfile
import urllib.parse
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import rasterio
from rasterio.crs import CRS
from rasterio.transform import from_origin
from rasterio.warp import transform as rio_transform
from rasterio.windows import from_bounds, Window
from rasterio.enums import Resampling

## 2) Configuration

Now, we will set up the configuration of all our global variables. 

In [ ]:
SITES = [
    "calaveras-big-trees",
    "dl-bliss",
    "independence-lake",
    "pacific-union-college",
    "sedgwick",
    "shaver-lake",
]

# Remote directory with files
BASE_ROOT = "https://wifire-data.sdsc.edu/nc/public.php/dav/files"

# Output directory
OUTPUT_ROOT = Path("./mask_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Keep this False to remove all temporary/intermediate files.
KEEP_TEMP = False

# Summary CSV of generated labels.
SAVE_SUMMARY_CSV = True

# Masking parameters
DEFAULT_RADIUS_M = 1.0
PAD_M = 20.0
BG_VAL = 0
SHRUB_VAL = 1
MASK_DTYPE = "uint8"

SESSION = requests.Session()

## 3) Small helpers

To ensure a smooth pipeline and avoid hardcoding, we have defined the following helpers:

- Identify the remote directory for each site
- Identify the corresponding **NAIP** `TIFF` file.
- Download each file.
- Clean up all files downloaded while executing the pipeline, as we are not interested in keeping them.

In [ ]:
def site_to_remote_base(site: str) -> str:
    return f"{BASE_ROOT}/ucca-{site}"

def site_to_tif_name(site: str) -> str:
    return site.replace("-", "_") + ".tif"

def download_file(url: str, dest: Path, chunk_size: int = 1024 * 1024) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    with SESSION.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
    return dest

def maybe_cleanup(path: Path):
    if path.exists():
        if path.is_dir():
            shutil.rmtree(path, ignore_errors=True)
        else:
            path.unlink(missing_ok=True)

## 4) WebDAV directory listing helpers

The following functions allow us to list the discovered files for each site instead of hardcoding the file names.

In [ ]:
def _strip_ns(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag

def list_webdav(url: str, depth: int = 1) -> list[dict]:
    headers = {
        "Depth": str(depth),
        "Content-Type": "application/xml",
    }
    body = '''<?xml version="1.0" encoding="utf-8" ?>
<d:propfind xmlns:d="DAV:">
  <d:prop>
    <d:resourcetype/>
    <d:getcontentlength/>
    <d:getlastmodified/>
  </d:prop>
</d:propfind>'''

    r = SESSION.request(
        "PROPFIND",
        url,
        headers=headers,
        data=body,
        timeout=120,
    )
    r.raise_for_status()

    root = ET.fromstring(r.text)
    entries = []
    for resp in root.iter():
        if _strip_ns(resp.tag) != "response":
            continue

        href = None
        is_collection = False
        content_length = None
        last_modified = None

        for child in resp.iter():
            tag = _strip_ns(child.tag)
            if tag == "href":
                href = child.text
            elif tag == "collection":
                is_collection = True
            elif tag == "getcontentlength" and child.text:
                try:
                    content_length = int(child.text)
                except ValueError:
                    content_length = None
            elif tag == "getlastmodified":
                last_modified = child.text

        if not href:
            continue

        decoded_href = urllib.parse.unquote(href)
        name = decoded_href.rstrip("/").split("/")[-1]
        if not name:
            continue

        entry_url = urllib.parse.urljoin(url.rstrip("/") + "/", urllib.parse.quote(name))
        entries.append(
            {
                "name": name,
                "url": entry_url,
                "is_dir": is_collection,
                "size": content_length,
                "last_modified": last_modified,
            }
        )

    dedup = {}
    for e in entries:
        dedup[(e["name"], e["is_dir"])] = e

    out = []
    base_name = urllib.parse.unquote(url.rstrip("/").split("/")[-1])
    for e in dedup.values():
        if e["name"] == base_name:
            continue
        out.append(e)

    out.sort(key=lambda x: (x["is_dir"], x["name"].lower()))
    return out

def list_files_with_suffix(url: str, suffixes: tuple[str, ...]) -> list[dict]:
    suffixes = tuple(s.lower() for s in suffixes)
    entries = list_webdav(url, depth=1)
    return [e for e in entries if (not e["is_dir"]) and e["name"].lower().endswith(suffixes)]

## 5) ALS metadata extraction

Before we can match any `TLS` shrub list to the right `ALS` tile, we need to know where each `ALS` file actually lives in space. These functions download each `ALS` file temporarily, pull out the key metadata using [**PDAL**](https://pdal.org/en/2.10.1/), and then discard the file.

In [ ]:
def trim_pdal_metadata(info: dict) -> dict:
    summary = {}
    md = info.get("metadata", {})
    stats = info.get("stats", {})

    summary["pdal_metadata_keys"] = sorted(md.keys())
    summary["stats_keys"] = sorted(stats.keys())

    for key in ["count", "compressed", "major_version", "minor_version", "dataformat_id", "srs"]:
        if key in md:
            summary[key] = md[key]

    native_bounds = {}
    for src_key, dst_key in [
        ("minx", "minx"), ("maxx", "maxx"),
        ("miny", "miny"), ("maxy", "maxy"),
        ("minz", "minz"), ("maxz", "maxz"),
    ]:
        if src_key in md:
            native_bounds[dst_key] = md[src_key]
    if native_bounds:
        summary["native_bounds"] = native_bounds

    if "srs" in md and isinstance(md["srs"], dict):
        srs = md["srs"]
        summary["srs_wkt"] = srs.get("compoundwkt") or srs.get("wkt")
        summary["srs_json"] = srs.get("json")

    for key in ["boundary", "bbox"]:
        if key in md:
            summary[key] = md[key]

    return summary

def extract_als_metadata(laz_path: Path) -> dict:
    cmd = ["pdal", "info", "--metadata", "--stats", str(laz_path)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"pdal info failed for {laz_path.name}:\n{r.stderr}")
    raw = json.loads(r.stdout)
    return trim_pdal_metadata(raw)

def build_als_metadata_for_site(site: str, temp_dir: Path) -> list[dict]:
    als_url = f"{site_to_remote_base(site)}/ALS"
    als_files = list_files_with_suffix(als_url, (".laz", ".las", ".copc.laz"))
    if not als_files:
        raise RuntimeError(f"No ALS files found for site '{site}' at {als_url}")

    metadata_records = []
    for entry in als_files:
        local_path = temp_dir / "als" / entry["name"]
        download_file(entry["url"], local_path)
        meta = extract_als_metadata(local_path)
        meta["source_file"] = entry["name"]
        metadata_records.append(meta)
        if not KEEP_TEMP:
            maybe_cleanup(local_path)

    return metadata_records

## 6) Transform and shrub-list helpers

Shrub coordinates come in `TLS` space, but our `ALS` tiles live in a different coordinate system. Transformation matrices (compact 4×4 grids of numbers) encode everything needed to move points from one space to the other: rotation, translation, and scale. These helpers parse those matrices from their `.txt` files and get the shrub lists ready for the conversion.

In [ ]:
def shrub_csv_to_transform_name(csv_name: str) -> str:
    stem = Path(csv_name).stem
    return f"{stem}toALS.txt"

def parse_transform_txt(txt_path: Path) -> np.ndarray:
    text = txt_path.read_text(errors="ignore")
    nums = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", text)
    vals = np.array([float(x) for x in nums], dtype=float)

    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    rows = []
    for ln in lines:
        row_nums = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", ln)
        if row_nums:
            rows.append([float(x) for x in row_nums])

    if len(rows) >= 4 and all(len(r) >= 4 for r in rows[:4]):
        return np.array([r[:4] for r in rows[:4]], dtype=float)

    if len(rows) >= 3 and all(len(r) >= 4 for r in rows[:3]):
        M = np.eye(4, dtype=float)
        M[:3, :] = np.array([r[:4] for r in rows[:3]], dtype=float)
        return M

    if vals.size == 16:
        return vals.reshape(4, 4)

    if vals.size == 12:
        M = np.eye(4, dtype=float)
        M[:3, :] = vals.reshape(3, 4)
        return M

    raise ValueError(f"Could not parse transform file: {txt_path.name}")

def apply_homogeneous_transform_xy(M4x4: np.ndarray, x: np.ndarray, y: np.ndarray, z0: float = 0.0):
    pts = np.stack([x, y, np.full_like(x, z0, dtype=float), np.ones_like(x, dtype=float)], axis=0)
    out = M4x4 @ pts
    return out[0], out[1], out[2]

def detect_xy_columns(df: pd.DataFrame) -> tuple[str, str]:
    cols = list(df.columns)
    lower = {c.lower(): c for c in cols}

    candidates = [
        ("x", "y"),
        ("x_utm", "y_utm"),
        ("easting", "northing"),
        ("utm_x", "utm_y"),
        ("xcoord", "ycoord"),
        ("lon", "lat"),
        ("longitude", "latitude"),
    ]
    for a, b in candidates:
        if a in lower and b in lower:
            return lower[a], lower[b]

    num_cols = [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]
    if len(num_cols) >= 2:
        return num_cols[0], num_cols[1]

    raise ValueError("Could not detect shrub X/Y columns.")

def detect_radius_column(df: pd.DataFrame) -> str | None:
    cols = list(df.columns)
    lower = {c.lower(): c for c in cols}
    candidates = [
        "radius", "radius_m", "r", "crown_radius", "shrub_radius",
        "buffer_m", "rad_m", "crownrad", "crown_radius_m",
    ]
    for c in candidates:
        if c in lower:
            return lower[c]
    return None

## 7) Tile matching and mask creation helpers

Once shrub coordinates are in `ALS` space, we need to figure out which `ALS` tile they belong to. We do that by counting how many shrubs fall within each tile's bounds and picking the best fit. From there, the following helpers take care of the rest: building a pixel grid aligned to the NAIP image, projecting each shrub into it, drawing a filled circle around each one, and writing the final result out as a georeferenced GeoTIFF.

In [ ]:
def get_tile_bounds_xy(meta: dict) -> tuple[float, float, float, float]:
    nb = meta.get("native_bounds")
    if not isinstance(nb, dict):
        raise ValueError(f"ALS metadata missing native_bounds for {meta.get('source_file')}")
    return float(nb["minx"]), float(nb["miny"]), float(nb["maxx"]), float(nb["maxy"])

def choose_best_als_tile(x_als: np.ndarray, y_als: np.ndarray, als_meta: list[dict]) -> dict:
    best = None
    best_count = -1
    for meta in als_meta:
        minx, miny, maxx, maxy = get_tile_bounds_xy(meta)
        inside = ((x_als >= minx) & (x_als <= maxx) & (y_als >= miny) & (y_als <= maxy))
        count = int(inside.sum())
        if count > best_count:
            best_count = count
            best = meta
    if best is None:
        raise RuntimeError("Could not choose an ALS tile.")
    return best

def naip_pixel_size_x(src) -> float:
    return float(abs(src.transform.a))

def naip_pixel_size_y(src) -> float:
    return float(abs(src.transform.e))

def build_aligned_grid_from_points(x_img: np.ndarray, y_img: np.ndarray, src) -> tuple:
    px = naip_pixel_size_x(src)
    py = naip_pixel_size_y(src)

    xmin = float(np.min(x_img) - PAD_M)
    xmax = float(np.max(x_img) + PAD_M)
    ymin = float(np.min(y_img) - PAD_M)
    ymax = float(np.max(y_img) + PAD_M)

    col0, row_top = (~src.transform) * (xmin, ymax)
    col1, row_bot = (~src.transform) * (xmax, ymin)

    col0 = math.floor(col0)
    row_top = math.floor(row_top)
    col1 = math.ceil(col1)
    row_bot = math.ceil(row_bot)

    width = int(col1 - col0)
    height = int(row_bot - row_top)
    if width <= 0 or height <= 0:
        raise ValueError("Computed output grid has non-positive dimensions.")

    x0, y0 = src.transform * (col0, row_top)
    dst_transform = from_origin(x0, y0, px, py)
    return dst_transform, width, height

def xy_to_rowcol(transform, x, y):
    inv = ~transform
    cols, rows = inv * (x, y)
    return np.asarray(rows), np.asarray(cols)

def draw_filled_circle(mask: np.ndarray, r0: int, c0: int, radius_px: int, value: int = 1):
    if radius_px < 1:
        radius_px = 1

    H, W = mask.shape
    rr_min = max(0, r0 - radius_px)
    rr_max = min(H, r0 + radius_px + 1)
    cc_min = max(0, c0 - radius_px)
    cc_max = min(W, c0 + radius_px + 1)

    yy, xx = np.ogrid[rr_min:rr_max, cc_min:cc_max]
    circle = (yy - r0) ** 2 + (xx - c0) ** 2 <= radius_px ** 2
    mask[rr_min:rr_max, cc_min:cc_max][circle] = value

def save_mask_geotiff(path: Path, arr: np.ndarray, crs, transform):
    profile = {
        "driver": "GTiff",
        "height": arr.shape[0],
        "width": arr.shape[1],
        "count": 1,
        "dtype": str(arr.dtype),
        "crs": crs,
        "transform": transform,
        "compress": "deflate",
        "nodata": BG_VAL,
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(arr, 1)

## 8) Per-shrub-list processing

With most of our helpers ready, we can now write the function that does the actual work: taking one shrub list at a time, running it through the full transformation and masking process, and producing a single mask file as output.

In [ ]:
def process_one_shrub_list(site: str, csv_entry: dict, naip_path: Path, als_meta: list[dict], temp_dir: Path) -> dict:
    shrub_local = temp_dir / "shrubs" / csv_entry["name"]
    download_file(csv_entry["url"], shrub_local)

    transform_name = shrub_csv_to_transform_name(csv_entry["name"])
    transform_url = f"{site_to_remote_base(site)}/transformations/{transform_name}"
    transform_local = temp_dir / "transforms" / transform_name
    download_file(transform_url, transform_local)

    df = pd.read_csv(shrub_local)
    x_col, y_col = detect_xy_columns(df)
    radius_col = detect_radius_column(df)

    x = pd.to_numeric(df[x_col], errors="coerce").to_numpy()
    y = pd.to_numeric(df[y_col], errors="coerce").to_numpy()

    valid_xy = np.isfinite(x) & np.isfinite(y)
    x = x[valid_xy]
    y = y[valid_xy]
    if x.size == 0:
        raise ValueError(f"{csv_entry['name']} has no valid shrub coordinates.")

    if radius_col is not None:
        radius_vals = pd.to_numeric(df.loc[valid_xy, radius_col], errors="coerce").fillna(DEFAULT_RADIUS_M).to_numpy()
        radius_vals = np.where(radius_vals > 0, radius_vals, DEFAULT_RADIUS_M)
    else:
        radius_vals = np.full(x.shape, DEFAULT_RADIUS_M, dtype=float)

    M = parse_transform_txt(transform_local)
    x_als, y_als, _ = apply_homogeneous_transform_xy(M, x, y)

    tile = choose_best_als_tile(x_als, y_als, als_meta)
    tile_wkt = tile.get("srs_wkt")
    if not tile_wkt:
        raise ValueError(f"ALS tile {tile.get('source_file')} is missing CRS/WKT metadata.")

    with rasterio.open(naip_path) as src:
        x_img, y_img = rio_transform(CRS.from_wkt(tile_wkt), src.crs, list(x_als), list(y_als))
        x_img = np.asarray(x_img, dtype=float)
        y_img = np.asarray(y_img, dtype=float)

        dst_transform, width, height = build_aligned_grid_from_points(x_img, y_img, src)
        rows, cols = xy_to_rowcol(dst_transform, x_img, y_img)

        px = naip_pixel_size_x(src)
        py = naip_pixel_size_y(src)
        avg_px_m = (px + py) / 2.0

        mask = np.full((height, width), BG_VAL, dtype=MASK_DTYPE)
        for r, c, radius_m in zip(rows, cols, radius_vals):
            radius_px = max(1, int(round(float(radius_m) / avg_px_m)))
            draw_filled_circle(mask, int(round(r)), int(round(c)), radius_px, value=SHRUB_VAL)

        out_dir = OUTPUT_ROOT / site
        out_path = out_dir / f"{Path(csv_entry['name']).stem}_mask.tif"
        save_mask_geotiff(out_path, mask, src.crs, dst_transform)

    if not KEEP_TEMP:
        maybe_cleanup(shrub_local)
        maybe_cleanup(transform_local)

    return {
        "site": site,
        "csv": csv_entry["name"],
        "mask_path": str(out_path),
        "n_points": int(x.size),
        "radius_column": radius_col if radius_col is not None else "",
        "als_tile": tile.get("source_file"),
        "mask_height": int(mask.shape[0]),
        "mask_width": int(mask.shape[1]),
    }

## 9) Per-site processing

With all the pieces in place, we can now tie everything together into a single function that runs the full workflow for one site at a time.

In [ ]:
def process_site(site: str) -> list[dict]:
    print(f"\n{'='*80}")
    print(f"Processing site: {site}")
    print(f"{'='*80}")

    site_base = site_to_remote_base(site)
    shrub_url = f"{site_base}/shrub_lists" # To use the revised version of the shrub lists, comment out this cell and uncomment the next one
    # shrub_url = f"{site_base}/shrub_lists_revised"
    naip_url = f"{site_base}/NAIP_3DEP_product/{site_to_tif_name(site)}"

    site_temp = Path(tempfile.mkdtemp(prefix=f"mask_pipeline_{site}_"))
    results = []

    try:
        shrub_files = list_files_with_suffix(shrub_url, (".csv",))
        if not shrub_files:
            raise RuntimeError(f"No shrub CSV files found for site '{site}' at {shrub_url}")
        print(f"Found {len(shrub_files)} shrub-list CSV files.")

        naip_local = site_temp / "naip" / site_to_tif_name(site)
        print(f"Downloading NAIP TIFF: {site_to_tif_name(site)}")
        download_file(naip_url, naip_local)

        print("Extracting ALS metadata with PDAL...")
        als_meta = build_als_metadata_for_site(site, site_temp)
        print(f"Built metadata for {len(als_meta)} ALS files.")

        for i, csv_entry in enumerate(shrub_files, start=1):
            print(f"[{i}/{len(shrub_files)}] {csv_entry['name']}")
            try:
                rec = process_one_shrub_list(site, csv_entry, naip_local, als_meta, site_temp)
                results.append(rec)
            except Exception as e:
                print(f"   ERROR -> {csv_entry['name']}: {e}")

        return results

    finally:
        if not KEEP_TEMP:
            maybe_cleanup(site_temp)

## 10) Environment check

As a final pre-pipeline check, we will verify that PDAL is available. If you see the message "PDAL is not available in this environment," it means that you started a server without the correct image, `pramonettivega/shrubs-labels:v1` (see module's instructions). You will then need to stop the current server and start it again. 

In [ ]:
v = subprocess.run(["pdal", "--version"], capture_output=True, text=True)
if v.returncode != 0:
    raise RuntimeError(
        "PDAL is not available in this environment. "
        f"stderr:\n{v.stderr}"
    )

print("PDAL OK:", (v.stdout or v.stderr).strip())

## 11) Run the full pipeline

Now that everything is in place, we're ready to run the full pipeline. Get a coffee because the next cell will take about 30–35 minutes. Once it's finished, all of the mask `TIFF` files will be saved under `mask_outputs`.

In [ ]:
all_results = []

for site in SITES:
    site_results = process_site(site)
    all_results.extend(site_results)

results_df = pd.DataFrame(all_results)
results_df

## 12) OPTIONAL: Save Summary CSV

To get a full summary of the mask files produced, run the next cell to produce a CSV file containing a list of all the files produced. 

In [ ]:
if SAVE_SUMMARY_CSV and not results_df.empty:
    summary_path = OUTPUT_ROOT / "mask_generation_summary.csv"
    results_df.to_csv(summary_path, index=False)
    print("Saved summary:", summary_path.resolve())
else:
    print("Summary CSV not saved.")

## 13) Deduplicate masks 

If you paid attention to how files are named, you might have noticed that they all follow a pattern. They start with a five-letter site code, followed by the plot number and the date of collection. 
For some plots, there are multiple collection dates. This leads to discrepancies in the produced shrub lists. To avoid duplication, we will keep the oldest dated file because the **NAIP** data was collected in 2020. 

In [ ]:
_MASK_RE = re.compile(
    r'^(?P<site>[A-Za-z]{5})_(?P<plot>\d{4})_(?P<date>\d{8})_\d+_mask\.tif$',
    re.IGNORECASE,
)

def deduplicate_masks(output_root: Path) -> None:
    """Walk every site sub-folder under *output_root* and remove duplicate
    masks, keeping only the one with the oldest (earliest) date string."""

    total_removed = 0

    # Iterate over immediate sub-directories 
    site_dirs = sorted(p for p in output_root.iterdir() if p.is_dir())

    if not site_dirs:
        print("No site sub-folders found under", output_root.resolve())
        return

    for site_dir in site_dirs:
        # Group mask files by (site_code, plot_id)
        groups: dict[tuple[str, str], list[Path]] = defaultdict(list)

        for tif in site_dir.glob("*.tif"):
            m = _MASK_RE.match(tif.name)
            if m:
                key = (m.group("site").upper(), m.group("plot"))
                groups[key].append(tif)

        removed_in_site = 0

        for (site_code, plot_id), files in groups.items():
            if len(files) <= 1:
                continue  # no duplicates

            # Sort by the date component extracted from the filename (ascending)
            files_sorted = sorted(
                files,
                key=lambda p: _MASK_RE.match(p.name).group("date"),
            )

            keeper = files_sorted[0]   # oldest date → keep
            to_remove = files_sorted[1:]  # newer dates → delete

            print(
                f"  [{site_dir.name}] plot {site_code}_{plot_id}: "
                f"keeping {keeper.name}, "
                f"removing {[f.name for f in to_remove]}"
            )

            for dup in to_remove:
                dup.unlink()
                removed_in_site += 1

        if removed_in_site:
            print(f"  → {removed_in_site} duplicate(s) removed in '{site_dir.name}'")
        else:
            print(f"  [{site_dir.name}] no duplicates found")

        total_removed += removed_in_site

    print(f"\nDeduplication complete. Total files removed: {total_removed}")


deduplicate_masks(OUTPUT_ROOT)


## 14) Visualization

We will do a quick visualization to check how the labels look atop of the NAIP tiles

In [ ]:
naip_local = Path("./naip/calaveras_big_trees.tif")  # Using 
naip_url = f"{BASE_ROOT}/ucca-calaveras-big-trees/NAIP_3DEP_product/calaveras_big_trees.tif"
Path("./naip").mkdir(parents=True, exist_ok=True)

download_file(naip_url, naip_local)

naip_path = "./naip/calaveras_big_trees.tif"
mask_path = "./mask_outputs/calaveras-big-trees/CATCU_0001_20240923_1_mask.tif" # We use a single example

In [ ]:
with rasterio.open(mask_path) as msrc:
    mask_bounds = msrc.bounds
    mask = msrc.read(1).astype(float)
    nodata = msrc.nodata

if nodata is not None:
    mask = np.where(mask == nodata, np.nan, mask)

# Check mask is inside NAIP
with rasterio.open(naip_path) as nsrc:
    nb = nsrc.bounds
    inside = (mask_bounds.left   >= nb.left  and
              mask_bounds.right  <= nb.right and
              mask_bounds.bottom >= nb.bottom and
              mask_bounds.top    <= nb.top)
    print(f"Mask inside NAIP bounds: {inside}")

    # Read only the NAIP window that corresponds to the mask extent
    win = from_bounds(*mask_bounds, transform=nsrc.transform)
    rgb = nsrc.read([1, 2, 3], window=win).astype(float)

# Normalize
for i in range(3):
    b = rgb[i]
    rgb[i] = (b - b.min()) / (b.max() - b.min() + 1e-9)
rgb_display = np.moveaxis(rgb, 0, -1)

# Resize mask to match the cropped NAIP window pixel size
from skimage.transform import resize
mask_resized = resize(mask, (rgb_display.shape[0], rgb_display.shape[1]), order=0, preserve_range=True)

overlay = np.zeros((*mask_resized.shape, 4), dtype=float)
overlay[mask_resized == 1] = [1.0, 0.1, 0.1, 0.6]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(rgb_display)
axes[0].set_title("NAIP crop at mask extent")
axes[0].axis("off")

axes[1].imshow(rgb_display)
axes[1].imshow(overlay)
axes[1].set_title("+ shrub mask overlay")
axes[1].axis("off")

plt.suptitle(f"CATCU_0001 — {mask_bounds}\n38.257°N, -120.221°W  |  {mask.shape[0]}x{mask.shape[1]} px @ 0.6m", fontsize=9)
plt.tight_layout()
plt.show()